In [ ]:
pip install requests beautifulsoup4 selenium

In [ ]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import os
import time
import random
from urllib.parse import urljoin

def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # Run in headless mode
    chrome_options.add_argument('--disable-gpu')
    return webdriver.Chrome(options=chrome_options)

def scrape_images(url, output_folder, max_images=50):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    driver = setup_driver()
    
    try:
        driver.get(url)
        # Scroll to load dynamic content
        last_height = driver.execute_script("return document.body.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)  # Wait for content to load
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
            
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        img_tags = soup.find_all('img')
        
        count = 0
        for img in img_tags:
            if count >= max_images:
                break
                
            # Get image URL
            img_url = img.get('src')
            if not img_url:
                continue
                
            # Make absolute URL if relative
            img_url = urljoin(url, img_url)
            
            # Skip small images and icons
            if 'icon' in img_url.lower() or 'logo' in img_url.lower():
                continue
                
            try:
                # Add random delay between requests
                time.sleep(random.uniform(1, 3))
                
                # Download image
                response = requests.get(img_url, timeout=10)
                if response.status_code == 200:
                    file_name = f"clothing_{count}.jpg"
                    file_path = os.path.join(output_folder, file_name)
                    
                    with open(file_path, 'wb') as f:
                        f.write(response.content)
                    
                    print(f"Downloaded: {file_name}")
                    count += 1
                    
            except Exception as e:
                print(f"Error downloading {img_url}: {str(e)}")
                continue
                
    except Exception as e:
        print(f"Error scraping website: {str(e)}")
        
    finally:
        driver.quit()

# Example usage
urls = [
    'https://example-clothing-store.com/womens-clothing',
    'https://example-clothing-store.com/mens-clothing'
]

output_folder = 'scraped_images'

for url in urls:
    scrape_images(url, output_folder)
